![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [145]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None) 
cc_apps.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


In [146]:
cc_apps.describe()

,2,7,10,12
count,690.000000,690.000000,690.00000,690.000000
mean,4.758725,2.223406,2.40000,1017.385507
std,4.978163,3.346513,4.86294,5210.102598
min,0.000000,0.000000,0.00000,0.000000
25%,1.000000,0.165000,0.00000,0.000000
50%,2.750000,1.000000,0.00000,5.000000
75%,7.207500,2.625000,3.00000,395.500000
max,28.000000,28.500000,67.00000,100000.000000


In [147]:
cc_apps = cc_apps.replace("?", np.NaN)

In [148]:
# Replace missing categorical values with the most frequent value and numerical values with mean
cc_apps = cc_apps.apply(
    lambda col: col.fillna(col.value_counts().index[0]) if col.dtypes == "object" else col.fillna(col.mean())
)

In [149]:
# Create dummies
cc_apps = pd.get_dummies(cc_apps, drop_first=True)

print("Shape of music_dummies: {}".format(cc_apps.shape))

Shape of music_dummies: (690, 383)


In [150]:
X = cc_apps.iloc[:, :-1].values
y = cc_apps.iloc[:, [-1]].values

In [151]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

In [152]:
scaler = StandardScaler()
rescaledX_train = scaler.fit_transform(X_train)
rescaledX_test = scaler.fit_transform(X_test)

In [153]:
# Instantiate the model
logreg = LogisticRegression()

# Fit the model
logreg.fit(rescaledX_train, y_train)

LogisticRegression()

In [154]:
y_train_pred = logreg.predict(rescaledX_train)

In [155]:
# Calculate the confusion matrix
print(confusion_matrix(y_train, y_train_pred))

[[203   3]
 [  3 253]]


In [156]:
from sklearn.metrics import classification_report

# Calculate the classification report
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       206
           1       0.99      0.99      0.99       256

    accuracy                           0.99       462
   macro avg       0.99      0.99      0.99       462
weighted avg       0.99      0.99      0.99       462



In [157]:
# Create the parameter space
params = {"tol": [0.01, 0.001, 0.0001],
         "max_iter": [100, 150, 200]}

In [158]:
# Instantiate the grid search object
grid_model = GridSearchCV(estimator=logreg, param_grid=params, cv=5)

In [159]:
# Fit to the training data
grid_model.fit(rescaledX_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(),
             param_grid={'max_iter': [100, 150, 200],
                         'tol': [0.01, 0.001, 0.0001]})

In [160]:
grid_model_result = grid_model.fit(rescaledX_train, y_train)

In [161]:
best_train_score, best_train_params = grid_model_result.best_score_, grid_model_result.best_params_

In [162]:
best_model = grid_model_result.best_estimator_
best_score =  best_model.score(rescaledX_test, y_test)

In [163]:
print(best_score)

0.8377192982456141
